In [ ]:
import os 
import cv2
import numpy as np
from tqdm import tqdm
import random
import torch
import torch.utils as utils
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import matplotlib.pyplot as plt
import torchvision.io
import torchvision.transforms as T
from PIL import Image
%matplotlib inline

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"# Using device: {device}")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
torch.cuda.empty_cache()

In [ ]:
class ClearHazyIterableDataset(IterableDataset):
    def __init__(self, clear_dir, hazy_dir, device, img_size=256, shuffle=False, clear_format="png", hazy_format="jpg"):
        self.clear_dir = clear_dir
        self.hazy_dir = hazy_dir
        self.device = device
        self.clear_files = sorted(os.listdir(clear_dir))
        self.hazy_files = sorted(os.listdir(hazy_dir))
        self.shuffle = shuffle
        self.clear_format = clear_format 
        self.hazy_format = hazy_format
        self.transform = T.Compose([
            T.ToTensor(), 
            T.Resize((img_size, img_size))
        ])

    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.hazy_files)
        for hazy_image_name in self.hazy_files:
            hazy_image_path = os.path.join(self.hazy_dir, hazy_image_name)
            clear_image_name = hazy_image_name.split('_')[0] + '.' + self.clear_format 
            clear_image_path = os.path.join(self.clear_dir, clear_image_name)

            hazy_image = Image.open(hazy_image_path).convert("RGB")
            clear_image = Image.open(clear_image_path).convert("RGB")
           
            hazy_image = self.transform(hazy_image).to(self.device)
            clear_image = self.transform(clear_image).to(self.device)

            yield clear_image, hazy_image

In [ ]:

clear_dir_its = 'RESIDE\ITS\clear'
hazy_dir_its = 'RESIDE\ITS\hazy'
clear_dir_ots = 'RESIDE\OTS\clear'
hazy_dir_ots = 'RESIDE\OTS\hazy'

In [ ]:

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.avg_pool(x)
        out = self.fc(out)
        return x * out


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        mask = self.conv(x_cat)
        mask = self.sigmoid(mask)
        return x * mask


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UpConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(UpConvBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_channels, out_channels, kernel_size, padding)

    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        return x

class SAUNet(nn.Module):
    def __init__(self):
        super(SAUNet, self).__init__()
        self.enc1 = ConvBlock(4, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)  

        # 通道注意力
        self.ca1 = ChannelAttention(32)
        self.ca2 = ChannelAttention(64)
        self.ca3 = ChannelAttention(128)
        self.ca4 = ChannelAttention(256) 

        # 空间注意力
        self.sa1 = SpatialAttention()
        self.sa2 = SpatialAttention()
        self.sa3 = SpatialAttention()
        self.sa4 = SpatialAttention() 
        
        self.dec1 = UpConvBlock(256, 128)  
        self.dec2 = ConvBlock(256, 128)
        self.dec3 = UpConvBlock(128, 64)
        self.dec4 = ConvBlock(128, 64)
        self.dec5 = UpConvBlock(64, 32)
        self.dec6 = ConvBlock(64, 32)
        self.dec7 = nn.Conv2d(32, 4, kernel_size=1)  

        # 处理 hazy_image 的卷积特征
        self.hazy_conv1 = ConvBlock(3, 32)
        self.hazy_conv2 = ConvBlock(32, 64)
        self.hazy_conv3 = ConvBlock(64, 128)  
        
    def forward(self, x, hazy_image):
        # 处理 hazy_image 的卷积特征
        hazy_features1 = self.hazy_conv1(hazy_image)
        hazy_features2 = self.hazy_conv2(hazy_features1)
        hazy_features3 = self.hazy_conv3(hazy_features2) 

        
        e1 = self.enc1(x)
        e1 = self.ca1(e1)  
        e1 = e1 + hazy_features1  

        e2 = self.enc2(e1)
        e2 = self.ca2(e2)  
        e2 = e2 + hazy_features2  

        e3 = self.enc3(e2)
        e3 = self.ca3(e3) 
        e3 = e3 + hazy_features3  

        e4 = self.enc4(e3)
        e4 = self.ca4(e4) 

        
        e4_att = self.sa4(e4)
        e3_att = self.sa3(e3)
        
        d1 = self.dec1(e4_att)
        d1 = F.interpolate(d1, size=(e3.size(2), e3.size(3)), mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat((d1, e3_att), dim=1))

        e2_att = self.sa2(e2)
        d2 = F.interpolate(d2, size=(e2.size(2), e2.size(3)), mode='bilinear', align_corners=False)
        d3 = self.dec3(d2)
        d3 = F.interpolate(d3, size=(e2.size(2), e2.size(3)), mode='bilinear', align_corners=False)
        d4 = self.dec4(torch.cat((d3, e2_att), dim=1))

        e1_att = self.sa1(e1)
        d4 = F.interpolate(d4, size=(e1.size(2), e1.size(3)), mode='bilinear', align_corners=False)
        d5 = self.dec5(d4)
        d5 = F.interpolate(d5, size=(e1.size(2), e1.size(3)), mode='bilinear', align_corners=False)
        d6 = self.dec6(torch.cat((d5, e1_att), dim=1))
       
        fog_layer = self.dec7(d6)
       
        return fog_layer
        
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.in1 = nn.InstanceNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.in2 = nn.InstanceNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.InstanceNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.in1(self.conv1(x)))
        out = self.in2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class Encoder(nn.Module):
    def __init__(self, in_channels):
        super(Encoder, self).__init__()
        self.block1 = ResidualConvBlock(in_channels, 64)
        self.block2 = ResidualConvBlock(64, 128)
        self.block3 = ResidualConvBlock(128, 256)
        self.block4 = ResidualConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x1 = self.block1(x)  # [batch_size, 64, 256, 256]
        x2 = self.pool(x1)   # [batch_size, 64, 128, 128]
        x3 = self.block2(x2)  # [batch_size, 128, 128, 128]
        x4 = self.pool(x3)   # [batch_size, 128, 64, 64]
        x5 = self.block3(x4)  # [batch_size, 256, 64, 64]
        x6 = self.pool(x5)   # [batch_size, 256, 32, 32]
        x7 = self.block4(x6)  # [batch_size, 512, 32, 32] 
        return x7, x5, x3, x1

class Decoder(nn.Module):
    def __init__(self, out_channels):
        super(Decoder, self).__init__()
        self.upconv1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2) 
        self.block1 = ResidualConvBlock(512, 256) 
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.block2 = ResidualConvBlock(256, 128)
        self.upconv3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.block3 = ResidualConvBlock(128, 64)
        self.conv_last = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x, enc_x5, enc_x3, enc_x1):
        x = self.upconv1(x)           # [batch_size, 256, 64, 64]
        x = torch.cat([x, enc_x5], dim=1)  # Concatenate with corresponding encoder output
        x = self.block1(x)            # [batch_size, 256, 64, 64]
        x = self.upconv2(x)           # [batch_size, 128, 128, 128]
        x = torch.cat([x, enc_x3], dim=1)  # Concatenate with corresponding encoder output
        x = self.block2(x)            # [batch_size, 128, 128, 128]
        x = self.upconv3(x)           # [batch_size, 64, 256, 256]
        x = torch.cat([x, enc_x1], dim=1)  # Concatenate with corresponding encoder output
        x = self.block3(x)            # [batch_size, 64, 256, 256]
        x = self.conv_last(x)         # [batch_size, 3, 256, 256]
        return x

class ResidualUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(ResidualUNet, self).__init__()
        self.encoder = Encoder(in_channels)
        self.decoder = Decoder(out_channels)

    def forward(self, x):
        # 解包 4 个值，而不是 3 个
        enc_out, enc_x5, enc_x3, enc_x1 = self.encoder(x)
        dec_out = self.decoder(enc_out, enc_x5, enc_x3, enc_x1)  
        return dec_out

class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()
        self.conv1 = nn.Conv2d(img_channels, 64, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)
        self.fc1 = nn.Linear(512 * 16 * 16, 1)  

        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.conv1(x)), 0.2)
        x = F.leaky_relu(self.bn2(self.conv2(x)), 0.2)
        x = F.leaky_relu(self.bn3(self.conv3(x)), 0.2)
        x = F.leaky_relu(self.bn4(self.conv4(x)), 0.2)
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        return x

In [ ]:
from torchviz import make_dot
import torch

# 生成一个示例输入张量
x1 = torch.randn(1, 4, 256, 256).requires_grad_(True).cuda() 
x2 = torch.randn(1, 3, 256, 256).requires_grad_(True).cuda() 
# 实例化模型
model1 = SAUNet().cuda()
model2 = ResidualUNet().cuda()
# 前向传播，获取输出
y1 = model1(x1,x2)
y2 = model2(x2)
# 绘制计算图
make_dot(y1, params=dict(list(model1.named_parameters()) + [('x', x1)])).render("SA-UNET", format="png")
make_dot(y2, params=dict(list(model2.named_parameters()) + [('x', x2)])).render("Residual_UNet", format="png")

In [ ]:
import torchvision.models as models
import torch.nn.functional as F
def compute_g_J(hazy_image, m_rgba):
    
    RGB = m_rgba[:, :3, :, :] 
    alpha = m_rgba[:, 3:4, :, :]  


    g_J = (hazy_image - RGB) / (1 - alpha + 1e-8) + RGB
    return g_J

def compute_I_rec(J_rec, m_rgba):
   
    RGB = m_rgba[:, :3, :, :] 
    alpha = m_rgba[:, 3:4, :, :]  

    # 计算 I_rec(x)
    I_rec = J_rec * (1 - alpha + 1e-8) + RGB * alpha
    return I_rec

In [ ]:
def compute_depth_from_histogram(I_hazy, I_clear):
    if I_hazy.dim() == 4 and I_clear.dim() == 4:
        gray_hazy = (0.2989 * I_hazy[:, 0] + 0.5870 * I_hazy[:, 1] + 0.1140 * I_hazy[:, 2]).detach()
        gray_clear = (0.2989 * I_clear[:, 0] + 0.5870 * I_clear[:, 1] + 0.1140 * I_clear[:, 2]).detach()

        hist_hazy = torch.histc(gray_hazy, bins=256, min=0, max=255)
        hist_clear = torch.histc(gray_clear, bins=256, min=0, max=255)

        hist_hazy /= hist_hazy.sum()
        hist_clear /= hist_clear.sum()

        hist_diff = torch.abs(hist_hazy - hist_clear)

        depth_map = 1 - hist_diff.sum()

        return depth_map
    else:
        raise ValueError("Input tensors must have shape (B, C, H, W).")

In [ ]:
class MaskGenerator(nn.Module):
    def __init__(self):
        super(MaskGenerator, self).__init__()
        
    def forward(self, x):
        batch_size = x.size(0)
        
        rgb_mask = torch.ones(batch_size, 3, IMG_SIZE, IMG_SIZE).cuda() 
        
        alpha_mask = torch.ones(batch_size, 1, IMG_SIZE, IMG_SIZE).cuda()
        
        rgba_mask = torch.cat((rgb_mask, alpha_mask), dim=1)
        
        return rgba_mask

In [ ]:

torch.autograd.set_detect_anomaly(True)
# 超参数设置
IMG_SIZE = 256
EPOCHS = 200
batch_size = 8
max_iterations = 1000
mse_loss_func = nn.MSELoss()
smooth_loss_func = nn.SmoothL1Loss()
BCE_loss_func = nn.BCELoss()

residual_unet = ResidualUNet().cuda()
sa_unet = SAUNet().cuda()
orig_discriminator = Discriminator().cuda()
hazy_discriminator = Discriminator().cuda()
mask_generator = MaskGenerator().cuda()

learning_rate_sa_unet = 0.0002  
learning_rate_residual_unet = 0.0002  
learning_rate_discriminator = 0.0005  
optimizer = torch.optim.Adam([
    {'params': sa_unet.parameters(), 'lr': learning_rate_sa_unet},
    {'params': residual_unet.parameters(), 'lr': learning_rate_residual_unet},
    {'params': orig_discriminator.parameters(), 'lr': learning_rate_discriminator},
    {'params': hazy_discriminator.parameters(), 'lr': learning_rate_discriminator}
])

losses = []
best_loss = float('inf')

In [ ]:

for epoch in range(EPOCHS):
    its_dataset = ClearHazyIterableDataset(clear_dir=clear_dir_its, hazy_dir=hazy_dir_its, device=device, img_size=IMG_SIZE, shuffle=True, clear_format="png", hazy_format="png")
    ots_dataset = ClearHazyIterableDataset(clear_dir=clear_dir_ots, hazy_dir=hazy_dir_ots, device=device, img_size=IMG_SIZE, shuffle=True, clear_format="jpg", hazy_format="jpg")
    
    its_loader = torch.utils.data.DataLoader(its_dataset, batch_size=batch_size)
    ots_loader = torch.utils.data.DataLoader(ots_dataset, batch_size=batch_size)

    its_iter = iter(its_loader)
    ots_iter = iter(ots_loader)

    for i in range(max_iterations):  
        if random.random() < 0.5:
            try:
                clear_image, hazy_image = next(its_iter) 
            except StopIteration:
                its_iter = iter(its_loader)  
                clear_image, hazy_image = next(its_iter)
        else:
            try:
                clear_image, hazy_image = next(ots_iter)  
            except StopIteration:
                ots_iter = iter(ots_loader) 
                clear_image, hazy_image = next(ots_iter)

  
        clear_image = Variable(clear_image).cuda()
        hazy_image = Variable(hazy_image).cuda()
        
        optimizer.zero_grad()

      
        input_features = torch.randn(batch_size, 256).cuda()  
        random_m = mask_generator(input_features)
        m_rgba = sa_unet(random_m, hazy_image)

      
        g_J = compute_g_J(hazy_image, m_rgba)

      
        J_rec = residual_unet(g_J)
        I_rec = compute_I_rec(J_rec, m_rgba)

       
        real_labels = torch.ones(batch_size, 1).cuda()
        fake_labels = torch.zeros(batch_size, 1).cuda()

    
        D_orig_real = orig_discriminator(clear_image)
        D_orig_fake = orig_discriminator(J_rec.detach())

        loss_D_orig_real = BCE_loss_func(D_orig_real, real_labels)
        loss_D_orig_fake = BCE_loss_func(D_orig_fake, fake_labels)
        loss_D_orig = (loss_D_orig_real + loss_D_orig_fake) / 2


        D_hazy_real = hazy_discriminator(hazy_image)
        D_hazy_fake = hazy_discriminator(I_rec.detach())

        loss_D_hazy_real = BCE_loss_func(D_hazy_real, real_labels)
        loss_D_hazy_fake = BCE_loss_func(D_hazy_fake, fake_labels)
        loss_D_hazy = (loss_D_hazy_real + loss_D_hazy_fake) / 2

 
        loss_D = (loss_D_orig + loss_D_hazy) / 2

      
        loss_D.backward(retain_graph=True)  
        optimizer.step()

      
        loss_J = mse_loss_func(J_rec, clear_image)
        loss_I = mse_loss_func(I_rec, hazy_image)

      
        D_orig_fake = orig_discriminator(J_rec)
        D_hazy_fake = hazy_discriminator(I_rec)

        loss_G_orig = BCE_loss_func(D_orig_fake, real_labels)
        loss_G_hazy = BCE_loss_func(D_hazy_fake, real_labels)

     
        deepth_loss = compute_depth_from_histogram(J_rec, clear_image)

        
        total_loss_G = loss_J * 2 + loss_I  + loss_G_orig + loss_G_hazy + deepth_loss * 2

    
        optimizer.zero_grad()
        total_loss_G.backward()
        optimizer.step()

      
        if total_loss_G.item() < best_loss:
            best_loss = total_loss_G.item()
            torch.save({
                'residual_unet_state_dict': residual_unet.state_dict(),
                'sa_unet_state_dict': sa_unet.state_dict(),
                'orig_discriminator_state_dict': orig_discriminator.state_dict(),
                'hazy_discriminator_state_dict': hazy_discriminator.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'losses': losses
            }, f'models/best_dehaze_model_epoch_{epoch+1}_batch_{i+1}.pth')
            print(f'模型已保存: Epoch {epoch+1}, Batch {i+1}')
            
    losses.append(total_loss_G.item())

    avg_loss = losses[epoch]
    print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}')
    
    if (epoch + 1) % 2 == 0:
        hazy_image_show = hazy_image[0].detach().cpu().numpy().transpose(1, 2, 0)
        clear_image_show = clear_image[0].detach().cpu().numpy().transpose(1, 2, 0)
        J_rec_show = J_rec[0].detach().cpu().numpy().transpose(1, 2, 0)
        I_rec_show = I_rec[0].detach().cpu().numpy().transpose(1, 2, 0)

        # 绘制图像
        fig, axs = plt.subplots(1, 4, figsize=(20, 5))
        axs[0].imshow(hazy_image_show)
        axs[0].set_title('Hazy Image')
        axs[1].imshow(clear_image_show)
        axs[1].set_title('Original Image')
        axs[2].imshow(J_rec_show)
        axs[2].set_title('J_rec (Dehazed)')
        axs[3].imshow(I_rec_show)
        axs[3].set_title('I_rec (Recovered)')

        for ax in axs:
            ax.axis('off')
        plt.show()

torch.save({
    'residual_unet_state_dict': residual_unet.state_dict(),
    'sa_unet_state_dict': sa_unet.state_dict(),
    'orig_discriminator_state_dict': orig_discriminator.state_dict(),
    'hazy_discriminator_state_dict': hazy_discriminator.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'losses': losses
}, 'models\dehaze_model_final.pth')

In [ ]:
plt.title('Loss Plot')
plt.xlabel('Epochs')
plt.ylabel('Value')

cpu_losses = [float(loss) for loss in losses]

plt.plot(cpu_losses)
plt.show()

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


residual_unet_param_count = count_parameters(residual_unet)
print(f"ResidualUNet 参数量: {residual_unet_param_count:,}")

sa_unet_param_count = count_parameters(sa_unet)
print(f"SAUNet 参数量: {sa_unet_param_count:,}")

orig_discriminator_param_count = count_parameters(orig_discriminator)
print(f"Original Discriminator 参数量: {orig_discriminator_param_count:,}")

hazy_discriminator_param_count = count_parameters(hazy_discriminator)
print(f"Hazy Discriminator 参数量: {hazy_discriminator_param_count:,}")

mask_generator_param_count = count_parameters(mask_generator)
print(f"Mask Generator 参数量: {mask_generator_param_count:,}")

In [ ]:
checkpoint = torch.load('models/best_dehaze_model_epoch_146_batch_64.pth')
residual_unet = ResidualUNet().cuda()
sa_unet = SAUNet().cuda()
residual_unet.load_state_dict(checkpoint['residual_unet_state_dict'])
sa_unet.load_state_dict(checkpoint['sa_unet_state_dict'])

In [ ]:
from skimage.metrics import structural_similarity as ssim
def output_psnr_mse(img_orig, img_out):
    squared_error = np.square(img_orig - img_out)
    mse = np.mean(squared_error)
    psnr = 10 * np.log10(1.0/math.sqrt(mse))
    return psnr

def mean_psnr_srgb(ref_mat, res_mat):
    n_blk, h, w, c = ref_mat.shape
    mean_psnr = 0
    for b in range(n_blk):
        ref_block = ref_mat[b, :, :, :]
        res_block = res_mat[b, :, :, :]
        ref_block = np.reshape(ref_block, (h, w, c))
        res_block = np.reshape(res_block, (h, w, c))
        psnr = output_psnr_mse(ref_block, res_block)
        mean_psnr += psnr
    return mean_psnr / n_blk

def mean_ssim_srgb(ref_mat, res_mat):
    n_blk, h, w, c = ref_mat.shape
    mean_ssim = 0
    for b in range(n_blk):
        ref_block = ref_mat[b, :, :, :]
        res_block = res_mat[b, :, :, :]
        ref_block = np.reshape(ref_block, (h, w, c))
        res_block = np.reshape(res_block, (h, w, c))

        win_size = min(h, w)
        if win_size % 2 == 0:
            win_size -= 1
        data_range = ref_block.max() - ref_block.min()

        ssim1 = ssim(ref_block, res_block, gaussian_weights=True, use_sample_covariance=False,
                     multichannel=True, win_size=win_size, channel_axis=2, data_range=data_range)
        mean_ssim += ssim1
    return mean_ssim / n_blk

def show_random_samples(hazy_image, clear_image, dehazed_image):
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(hazy_image)
    plt.axis('off')
    plt.title('Hazy')

    plt.subplot(1, 3, 2)
    plt.imshow(clear_image)
    plt.axis('off')
    plt.title('Ground Truth')

    plt.subplot(1, 3, 3)
    plt.imshow(dehazed_image)
    plt.axis('off')
    plt.title('Dehazed')

    plt.tight_layout()
    plt.show()

In [ ]:
import math
# 图像转换
transform = T.Compose([
    T.ToTensor(),
    T.Resize((256, 256))
])

indoor_dir = 'RESIDE\SOTS\indoor'
outdoor_dir = 'RESIDE\SOTS\outdoor'
dense_dir = 'Dense_Haze_NTIRE19'

In [ ]:
def dehaze_and_evaluate(input_dir, clear_dir):
    hazy_files = os.listdir(input_dir)
    clear_files = os.listdir(clear_dir)
    
    clear_images = []
    dehazed_images = []
    hazy_images = [] 

    count = 0  
    for hazy_file in hazy_files:
        hazy_path = os.path.join(input_dir, hazy_file)
        clear_file = hazy_file.split('_')[0] + '.png'  
        clear_path = os.path.join(clear_dir, clear_file)

        if not os.path.exists(clear_path):
            print(f"未找到匹配的 clear 图像: {clear_file}")
            continue

       
        hazy_image = Image.open(hazy_path).convert('RGB')
        clear_image = Image.open(clear_path).convert('RGB')

     
        hazy_tensor = transform(hazy_image).unsqueeze(0).cuda()
        clear_tensor = transform(clear_image).unsqueeze(0).cuda()

        hazy_tensor = Variable(hazy_tensor).cuda()
        clear_tensor = Variable(clear_tensor).cuda()
        
     
        with torch.no_grad():
            input_features = torch.randn(1, 256).cuda()  
            random_m = mask_generator(input_features)
            m_rgba = sa_unet(random_m, hazy_tensor)
            g_J = compute_g_J(hazy_tensor, m_rgba)
            J_rec = residual_unet(g_J)

        dehazed_image_np = J_rec.squeeze(0).cpu().numpy().transpose(1, 2, 0)  # (H, W, C)
        clear_image_np = clear_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)  # (H, W, C)
        hazy_image_np = hazy_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)  # (H, W, C)

        if count % 50 == 0:
            show_random_samples(hazy_image_np, clear_image_np, dehazed_image_np)

        dehazed_image_np = dehazed_image_np.astype('float') / 255.0
        clear_image_np = clear_image_np.astype('float') / 255.0
        hazy_image_np = hazy_image_np.astype('float') / 255.0

 
        clear_images.append(clear_image_np)
        dehazed_images.append(dehazed_image_np)
        hazy_images.append(hazy_image_np)

        count += 1  

  
    print(f"处理 {input_dir} 的图像数量: {count}")


    clear_images = np.array(clear_images)
    dehazed_images = np.array(dehazed_images)
    hazy_images = np.array(hazy_images)  

    
    avg_ssim = mean_ssim_srgb(clear_images, dehazed_images)
    avg_psnr = mean_psnr_srgb(clear_images, dehazed_images)

    return avg_ssim, avg_psnr

In [ ]:

indoor_ssim, indoor_psnr = dehaze_and_evaluate(os.path.join(indoor_dir, 'hazy'), os.path.join(indoor_dir, 'gt'))
outdoor_ssim, outdoor_psnr = dehaze_and_evaluate(os.path.join(outdoor_dir, 'hazy'), os.path.join(outdoor_dir, 'gt'))

print(f'Indoor - Avg SSIM: {indoor_ssim:.4f}, Avg PSNR: {indoor_psnr:.4f}')
print(f'Outdoor - Avg SSIM: {outdoor_ssim:.4f}, Avg PSNR: {outdoor_psnr:.4f}')